# 📊 Turkey at the Olympics — Statistical Analysis

**Goal:** Apply statistical tests to validate findings and benchmark Turkey against peer nations.  
**Data:** Cleaned dataset from `01_EDA_cleaning.ipynb`

---

## 1. Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy import stats

sns.set_theme(style='whitegrid')
%matplotlib inline

df = pd.read_csv('../data/athlete_events_clean.csv')
df_turkey = df[(df['Team'] == 'Turkey') & (df['Season'] == 'Summer')].copy()

print('Data loaded ✅')

## 2. Hypothesis Test 1 — Age Difference: Medalists vs Non-Medalists

**H0:** There is no significant age difference between medalists and non-medalists.  
**H1:** Medalists are significantly different in age from non-medalists.

In [ ]:
medalists     = df_turkey[df_turkey['Total'] > 0]['Age'].dropna()
non_medalists = df_turkey[df_turkey['Total'] == 0]['Age'].dropna()

t_stat, p_value = stats.ttest_ind(medalists, non_medalists)

print('=== Independent T-Test: Age ===')
print(f'Medalists     — Mean Age: {medalists.mean():.2f} | Std: {medalists.std():.2f} | N: {len(medalists)}')
print(f'Non-Medalists — Mean Age: {non_medalists.mean():.2f} | Std: {non_medalists.std():.2f} | N: {len(non_medalists)}')
print(f'\nt-statistic: {t_stat:.4f}')
print(f'p-value:     {p_value:.4f}')
print(f'\nResult: {"✅ Statistically significant (p < 0.05)" if p_value < 0.05 else "❌ Not significant (p >= 0.05)"}')

In [ ]:
# Visualize
plt.figure(figsize=(9, 4))
sns.boxplot(
    data=df_turkey.assign(Group=df_turkey['Total'].apply(lambda x: 'Medalist' if x > 0 else 'Non-Medalist')),
    x='Group', y='Age',
    palette={'Medalist': '#FFD700', 'Non-Medalist': '#aaaaaa'}
)
plt.title('Age Distribution: Turkish Medalists vs Non-Medalists', fontsize=13)
plt.tight_layout()
plt.savefig('../visuals/ttest_age.png', dpi=150)
plt.show()

## 3. Hypothesis Test 2 — Medal Trend Over Time

**H0:** There is no correlation between year and total medals.  
**H1:** Turkey's medal count is correlated with time (improving performance).

In [ ]:
yearly = df_turkey.groupby('Year')['Total'].sum().reset_index()

corr, p_corr = stats.pearsonr(yearly['Year'], yearly['Total'])

print('=== Pearson Correlation: Year vs Total Medals ===')
print(f'Correlation coefficient (r): {corr:.4f}')
print(f'p-value:                     {p_corr:.4f}')
print(f'\nResult: {"✅ Significant positive correlation" if corr > 0 and p_corr < 0.05 else "❌ No significant correlation"}')

In [ ]:
# Scatter with regression line
plt.figure(figsize=(10, 4))
sns.regplot(data=yearly, x='Year', y='Total', 
            scatter_kws={'color': 'steelblue', 's': 60},
            line_kws={'color': 'red', 'linewidth': 1.5})
plt.title(f'Turkey Medal Trend Over Time (r = {corr:.3f}, p = {p_corr:.3f})', fontsize=13)
plt.tight_layout()
plt.savefig('../visuals/correlation_trend.png', dpi=150)
plt.show()

## 4. Hypothesis Test 3 — Height Difference by Medal Status

In [ ]:
height_med     = df_turkey[df_turkey['Total'] > 0]['Height'].dropna()
height_non_med = df_turkey[df_turkey['Total'] == 0]['Height'].dropna()

t2, p2 = stats.ttest_ind(height_med, height_non_med)

print('=== T-Test: Height ===')
print(f'Medalists     — Mean Height: {height_med.mean():.2f} cm')
print(f'Non-Medalists — Mean Height: {height_non_med.mean():.2f} cm')
print(f'p-value: {p2:.4f}')
print(f'Result: {"✅ Significant" if p2 < 0.05 else "❌ Not significant"}')

## 5. Turkey vs Peer Nations

In [ ]:
competitors = ['Turkey', 'Greece', 'Iran', 'Bulgaria', 'Hungary']
df_comp = df[(df['Team'].isin(competitors)) & (df['Season'] == 'Summer')]

comp_yearly = df_comp.groupby(['Year', 'Team'])['Total'].sum().reset_index()

fig = px.line(
    comp_yearly, x='Year', y='Total', color='Team',
    title='Turkey vs Peer Nations — Medal Trend',
    markers=True,
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.show()

In [ ]:
# Total medals comparison
comp_total = (df_comp.groupby('Team')[['Gold', 'Silver', 'Bronze', 'Total']]
              .sum()
              .sort_values('Total', ascending=False)
              .reset_index())

fig2 = px.bar(
    comp_total, x='Team', y=['Gold', 'Silver', 'Bronze'],
    title='Total Medals: Turkey vs Peer Nations (All Time)',
    color_discrete_map={'Gold': '#FFD700', 'Silver': '#C0C0C0', 'Bronze': '#CD7F32'},
    barmode='stack'
)
fig2.show()

## 6. One-Way ANOVA — Medal Count by Sport

In [ ]:
# Test: Is there a significant difference in medal counts between sports?
sport_groups = [
    group['Total'].values
    for _, group in df_turkey[df_turkey['Total'] > 0].groupby('Sport')
    if len(group) >= 5
]

f_stat, p_anova = stats.f_oneway(*sport_groups)

print('=== One-Way ANOVA: Medal Count by Sport ===')
print(f'F-statistic: {f_stat:.4f}')
print(f'p-value:     {p_anova:.4f}')
print(f'Result: {"✅ Significant difference between sports" if p_anova < 0.05 else "❌ No significant difference"}')

---
## ✅ Statistical Summary

| Test | Variables | Result |
|------|-----------|--------|
| T-Test | Age: Medalists vs Non-Medalists | See output |
| T-Test | Height: Medalists vs Non-Medalists | See output |
| Pearson Correlation | Year vs Total Medals | See output |
| One-Way ANOVA | Medal Count by Sport | See output |

**Next:** `04_ml_model.ipynb` → Medal prediction with Random Forest